## Outpu parsers :
### 1. JsonOutputParser
### 2. PydanticOutputParser
### 3. stringOutputParser
### 4. StructuredOutputParser

### Output Parsing : 

helps in convert raw llm reponse into structured data that can be easily used for further processing or analysis. It involves analyzing the output generated by a language model and extracting relevant information, patterns, or insights from it. This can be done using various techniques such as regular expressions, natural language processing, or custom parsing algorithms depending on the complexity and format of the output. The goal of output parsing is to transform unstructured data into a more organized and meaningful format that can be easily understood and utilized for decision-making or other applications.


Output parsing is the process of interpreting and extracting meaningful information from the output generated by a program or system. It involves analyzing the output data, identifying relevant patterns, and converting it into a structured format that can be easily understood and utilized for further processing or decision-making. This is often done using various techniques such as regular expressions, natural language processing, or custom parsing algorithms depending on the complexity and format of the output.

## Output Parsers :


### 1. string output parser :

This is the most basic type of output parser that simply returns the raw string output generated by a language model. It does not perform any additional processing or formatting on the output, allowing users to access the unaltered response directly. This can be useful in cases where users want to see the original output without any modifications or when they want to perform their own custom parsing on the raw string data.

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate, load_prompt
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ChatMessage
from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel, EmailStr, Field



load_dotenv()

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=1.3, max_tokens=100)



#  detailed report 1st prompt

template1 = PromptTemplate(

    template="You are a helpful assistant that provides detailed reports on the given topic. Please provide a comprehensive analysis and insights on the following topic: {topic}",

    input_variables=["topic"]

)

# 2nd prompt summery

template2 = PromptTemplate(

    template="You are a helpful assistant that provides concise summaries on the given topic. Please provide a brief summary and key takeaways on the following topic: {topic}",

    input_variables=["topic"]

)

prompt1 = template1.invoke({"topic": "The impact of climate change on global agriculture."})



result1 = model.invoke(prompt1)



prompt2 = template2.invoke({"topic": "The impact of climate change on global agriculture."})

result2 = model.invoke(prompt2)

print("Detailed Report:\n", result1)

print("Detailed Report:\n", result2)

#### NUSING STRING output parsing to extract key insights from the detailed report

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate, load_prompt
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser





load_dotenv()

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=1.3, max_tokens=100)



prompt1 = template1.invoke({"topic": "The impact of climate change on global agriculture."})



result1 = model.invoke(prompt1)



prompt2 = template2.invoke({"topic": "The impact of climate change on global agriculture."})

result2 = model.invoke(prompt2)



parser = StrOutputParser()



chain = template1 | model | parser | template2 | model | parser



result = chain.invoke({"topic": "The impact of climate change on global agriculture."})



print("Detailed Report:\n", result)

#### json output parsering to extract key insights from the detailed report

In [ ]:
parser = JsonOutputParser()



template = PromptTemplate(

    template = 'give me the name, age and city of a fictional person\n {format_instructions}\n',

    input_variables = [],

    partial_variables = {"format_instructions": parser.get_format_instructions()}

)



# prompt = template.format()



# result = model.invoke(prompt)



# parser.parse(result.content)



chain = template | model | parser



result = chain.invoke({})



print("Parsed Output:\n", result)

json output parser does not give output in a structured format, it gives output in string format which can be easily converted into a structured format using json.loads() method in python. This allows users to easily access and manipulate the data contained within the JSON output, making it more convenient for further processing or analysis. By using a JSON output parser, users can efficiently extract key insights from the detailed report generated by a language model, enabling them to make informed decisions based on the extracted information.

In [25]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

load_dotenv()
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=1.3, max_tokens=1000)


class Facts(BaseModel):
    fact_1: str = Field(description="Fact 1 about the topic \n")
    fact_2: str = Field(description="Fact 2 about the topic \n")
    fact_3: str = Field(description="Fact 3 about the topic \n")
    fact_4: str = Field(description="Fact 4 about the topic \n")
    fact_5: str = Field(description="Fact 5 about the topic \n")

parser = JsonOutputParser(pydantic_object=Facts)

template = PromptTemplate(
    template='Give 5 fact about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

chain = template | model | parser

result = chain.invoke({'topic':'black hole'})

print(result)

{'fact_1': 'Black holes are regions in space where the gravitational pull is so strong that nothing, including light, can escape.', 'fact_2': 'Black holes are formed when a massive star collapses in on itself and its gravity becomes so strong that it warps the fabric of spacetime.', 'fact_3': "The point of no return around a black hole is called the event horizon, and once something crosses the event horizon, it is trapped by the black hole's gravity.", 'fact_4': 'Black holes come in a range of sizes, from small, stellar-mass black holes formed from the collapse of individual stars, to supermassive black holes found at the centers of galaxies, with masses millions or even billions of times that of the sun.', 'fact_5': "The gravitational pull of a black hole is so strong that it can distort the fabric of spacetime around it, creating strange visual effects such as gravitational lensing, where the light from distant objects is bent and distorted by the black hole's gravity."}


### issue : 
we can't validate the output generated by the language model, as it may not always follow the expected format or structure. This can lead to errors or inconsistencies in the extracted insights, making it difficult to rely on the output for decision-making or analysis. Additionally, if the output is not properly validated, it may contain irrelevant or misleading information that can further complicate the process of extracting meaningful insights from the detailed report generated by a language model.
in structured output parsing is that it can be difficult to extract key insights from the detailed report generated by a language model. This is because the output may contain a large amount of information, making it challenging to identify the most relevant and important insights. Additionally, the output may be unstructured or contain noise, which can further complicate the process of extracting meaningful insights. As a result, users may need to spend additional time and effort analyzing the output to identify the key insights that are most relevant to their needs.

### Solution : 
using pydantic models to define the structure of the expected output and then using a JSON output parser to extract the relevant information from the detailed report generated by the language model. This approach allows users to easily access and manipulate the data contained within the JSON output, making it more convenient for further processing or analysis. By using pydantic models and a JSON output parser, users can efficiently extract key insights from the detailed report generated by a language model, enabling them to make informed decisions based on the extracted information.

In [29]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

load_dotenv()
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=1.3, max_tokens=1000)

class Person(BaseModel):

    name: str = Field(description='Name of the person')
    age: int = Field(gt=18, description='Age of the person')
    city: str = Field(description='Name of the city the person belongs to')

parser = PydanticOutputParser(pydantic_object=Person)

template = PromptTemplate(
    template='Generate the name, age and city of a fictional {place} person \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

chain = template | model | parser

final_result = chain.invoke({'place':'sri lankan'})

print(final_result)

name='Kavinda Perera' age=30 city='Colombo'


# Chains : 
chians in langchain are a sequence of calls to language models or other utilities, where the output of one call is used as the input for the next call. This allows users to create complex workflows and processes that can be easily managed and executed. Chains can be used to perform a variety of tasks, such as data processing, natural language understanding, and decision-making, by chaining together multiple steps that leverage the capabilities of language models and other tools. By using chains, users can automate and streamline their workflows, making it easier to achieve their desired outcomes efficiently and effectively.

# Runnable :
Runnable is a class in langchain that represents a unit of work that can be executed. It provides a standardized interface for defining and running tasks, allowing users to easily create and manage their workflows. A Runnable can be a simple function, a more complex class with multiple methods, or even a chain of tasks. By using the Runnable class, users can encapsulate their logic and functionality in a reusable and modular way, making it easier to maintain and extend their codebase. The Runnable class also provides features such as error handling, logging, and input/output management, allowing users to create robust and reliable applications that can be easily executed and monitored.

In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=1.3, max_tokens=1000)

prompt = PromptTemplate(
    template = 'Generate a creative story about a {topic}.',
    input_variables = ['topic']
)

parser = StrOutputParser()

chain = prompt | model | parser

result = chain.invoke({'topic':'cricket'})
print(result)

chain.get_graph().print_ascii()

Once upon a time, in a lush meadow nestled between two great oak trees, there lived a tiny cricket named Melodia. Melodia was no ordinary cricket – she had a voice like honey and gold, and when she sang, the stars shone brighter and the flowers bloomed more vibrantly.

Every night, Melodia would perch on a delicate blade of grass and serenade the meadow with her enchanting songs. The other creatures of the meadow would gather around her, entranced by the sweetness of her melody. Fireflies would twinkle in time with her rhythm, and rabbits would sway to the beat. Even the wind would whisper its own gentle harmony, weaving in and out of Melodia's trills.

As the nights passed, Melodia's fame spread far and wide. Creatures from neighboring meadows and forests would travel to hear the cricket's magical voice. A wise old owl named Professor Hootenanny, who lived in a hollow tree nearby, became Melodia's greatest fan. He would attend every performance, taking notes and analyzing the cricket'

### 1.Sequential Chain :
In a sequential chain, the output of one step is directly passed as the input to the next step. This creates a linear flow of data and allows for a straightforward execution of tasks. Each step in the chain is executed in order, and the output of one step can be used to inform or influence the next step. This type of chain is useful when there is a clear and logical progression of tasks that need to be performed in a specific sequence.
### 2.Parallel Chain :
In a parallel chain, multiple steps are executed simultaneously, allowing for faster processing and improved efficiency. Each step in the chain can be independent of the others, and the outputs from each step can be combined or used separately as needed. This type of chain is useful when there are tasks that can be performed concurrently, such as data processing or analysis that can be done in parallel to save time.
### 3.Conditional Chain :
In a conditional chain, the flow of execution is determined by certain conditions or criteria. Depending on the outcome of a specific step, the chain can branch off into different paths, allowing for more dynamic and flexible workflows. This type of chain is useful when there are multiple possible outcomes or when the execution of certain steps depends on the results of previous steps. By using conditional chains, users can create more complex and adaptable workflows that can handle a variety of scenarios and outcomes.
### 4.Recursive Chain :
In a recursive chain, a step can call itself or another step in the chain, allowing for repeated execution of certain tasks until a specific condition is met. This type of chain is useful when there are tasks that require iterative processing or when there is a need to handle nested or hierarchical data structures. By using recursive chains, users can create more sophisticated workflows that can handle complex data and perform tasks that require multiple iterations or levels of processing.
### 5.Parallel Conditional Chain :
In a parallel conditional chain, multiple steps are executed simultaneously, and the flow of execution is determined by certain conditions or criteria. This allows for a more dynamic and flexible workflow, where different paths can be taken based on the outcomes of the parallel steps. This type of chain is useful when there are tasks that can be performed concurrently, but the execution of certain steps depends on the results of the parallel steps. By using parallel conditional chains, users can create more efficient and adaptable workflows that can handle a variety of scenarios and outcomes while still benefiting from the advantages of parallel processing.
### 6. Recursive Conditional Chain :
In a recursive conditional chain, a step can call itself or another step in the chain based on certain conditions or criteria. This allows for repeated execution of certain tasks until a specific condition is met, while also providing the flexibility to branch off into different paths based on the outcomes of the recursive steps. This type of chain is useful when there are tasks that require iterative processing or when there is a need to handle nested or hierarchical data structures, while also allowing for dynamic decision-making based on the results of the recursive steps. By using recursive conditional chains, users can create more sophisticated and adaptable workflows that can handle complex data and perform tasks that require multiple iterations or levels of processing, while also providing the ability to make decisions based on the outcomes of the recursive steps.

In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=1.3, max_tokens=1000)

prompt1 = PromptTemplate(
    template = 'Generate a creative story about a {topic}.',
    input_variables = ['topic']
)

prompt2 = PromptTemplate(
    template = 'Summarize the following story in one sentence: {story}',
    input_variables = ['story']
)

parser = StrOutputParser()

chain = prompt1 | model | parser | prompt2 | model | parser

result = chain.invoke({'topic':'cricket'})
print(result)

chain.get_graph().print_ascii()

Croaky, a charismatic cricket with a magical voice, accepted the Moon Queen's invitation to perform at her celestial palace and, with his enchanting melodies, captivated the Queen and the stars, earning the title of Royal Cricket Composer and inspiring generations to come.
     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput | 

## Parallel chain

In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv

from langchain_core.runnables import RunnableParallel

load_dotenv()

model1 = ChatGroq(model="llama-3.3-70b-versatile", temperature=1.3, max_tokens=1000)

# 1. Define the LLM
model2 = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=1.3,
    max_tokens=1000
)


# 2. Explicitly pass the model_id to ChatHuggingFace
# This prevents the StopIteration error by skipping auto-detection
# model2 = ChatHuggingFace(llm=model2)

prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)

prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)

prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

merge_chain = prompt3 | model1 | parser

chain = parallel_chain | merge_chain

text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

result = chain.invoke({'text':text})

print(result)

chain.get_graph().print_ascii()


Support Vector Machines (SVMs) Notes and Quiz

### Advantages of SVMs

The following are the key advantages of using Support Vector Machines:

1. **Effective in high dimensional spaces**: SVMs are capable of handling high-dimensional data and can perform well even when the number of dimensions exceeds the number of samples.
2. **Memory efficient**: SVMs are memory efficient because they use a subset of training points (called support vectors) in the decision function, which reduces the computational requirements.
3. **Versatile with different kernel functions**: SVMs can be used with different kernel functions, allowing for flexibility in the type of relationships that can be modeled.
4. **Works well with more dimensions than samples**: SVMs are particularly effective in situations where there are more features than samples, making them a popular choice in applications involving high-dimensional data.

### Disadvantages of SVMs

The following are some of the limitations of using Suppor

## Conditional Chain

In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
from pydantic import BaseModel, Field
from typing import Literal



load_dotenv()

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=1.3, max_tokens=1000)

parser = StrOutputParser()

class Feedback(BaseModel):

    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into postive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier_chain | branch_chain

print(chain.invoke({'feedback': 'This is a beautiful phone'}))
print(chain.invoke({'feedback': 'This is a bad phone'}))

chain.get_graph().print_ascii()

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Thank you so much for your kind words. I'm glad to hear that you're pleased. If you have any other questions or need further assistance, don't hesitate to reach out. I'm here to help.
"I'm sorry to hear that you had a negative experience. Can you please provide more details about what went wrong? We take all feedback seriously and would like to make things right. Your input will help us to improve and provide a better experience for you and others in the future. Thank you for taking the time to share your concerns."
    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
      +----------+       
      | ChatGroq |       
      +----------+       
            *            
            *            
            *            
+---------

# Runnable :
used to define a unit of work that can be executed, allowing users to easily create and manage their workflows. A Runnable can be a simple function, a more complex class with multiple methods, or even a chain of tasks. By using the Runnable class, users can encapsulate their logic and functionality in a reusable and modular way, making it easier to maintain and extend their codebase. The Runnable class also provides features such as error handling, logging, and input/output management, allowing users to create robust and reliable applications that can be easily executed and monitored.

### Types of Runnable :
1. Function Runnable : A simple function that can be executed as a unit of work.
2. Class Runnable : A more complex class with multiple methods that can be executed as a unit of work.
3. Chain Runnable : A chain of tasks that can be executed as a unit of work, allowing for more complex workflows and processes to be defined and executed. By using the Runnable class, users can create reusable and modular code that can be easily maintained and extended, while also providing features such as error handling and logging to ensure the reliability of their applications.
### Task Specific Runnable : 
A Runnable that is designed to perform a specific task or function, such as data processing, natural language understanding, or decision-making. This type of Runnable can be used to encapsulate the logic and functionality required to perform a specific task, making it easier to manage and execute within a larger workflow or process. By using task-specific Runnables, users can create more focused and efficient code that is tailored to the specific needs of their applications, while still benefiting from the features and capabilities provided by the Runnable class.
### Primitive Runnable :
1.runnable sequential : A simple function that can be executed as a unit of work, allowing for straightforward execution of tasks in a linear flow.

2.runnable parallel : A more complex class with multiple methods that can be executed simultaneously, allowing for faster processing and improved efficiency in handling concurrent tasks.

3.runnable pass-through : A chain of tasks that simply passes the output from one step to the next without any additional processing or transformation, allowing for a straightforward flow of data through the chain.

4.runnable conditional : A chain of tasks that can branch off into different paths based on certain conditions or criteria, providing dynamic and flexible workflows that can handle a variety of scenarios and outcomes.

5.runnable recursive : A step that can call itself or another step in the chain, allowing for repeated execution of certain tasks until a specific condition is met, making it useful for handling iterative processing or nested data structures.

6.runnable parallel conditional : Multiple steps executed simultaneously, with the flow of execution determined by certain conditions or criteria, allowing for efficient and adaptable workflows that can handle a variety of scenarios and outcomes while benefiting from parallel processing.

7.runnable recursive conditional : A step that can call itself or another step in the chain based on certain conditions or criteria, allowing for repeated execution of certain tasks until a specific condition is met, while also providing the flexibility to branch off into different paths based on the outcomes of the recursive steps, making it useful for handling complex data and performing tasks that require multiple iterations or levels of processing, while also providing the ability to make decisions based on the outcomes of the recursive steps.

In [4]:
# 1. runnable sequence

from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnableSequence


load_dotenv()

model = ChatGroq(model="llama-3.3-70b-versatile")

prompt1 = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Explain the following joke - {text}',
    input_variables=['text']
)

parser = StrOutputParser()

chain = RunnableSequence(prompt1 | model | parser | prompt2 | model | parser)

print(chain.invoke({'topic':'AI'}))


A classic play on words. This joke is funny because it's a clever pun that exploits the multiple meanings of the word "process" in the context of artificial intelligence (AI) and human emotions.

In computing, "to process" means to perform operations on data, execute instructions, or handle information. AI programs are designed to process vast amounts of data, recognize patterns, and make decisions based on that data.

However, in the context of human emotions, "to process" means to deal with, understand, or come to terms with one's feelings. When someone is struggling to process their emotions, it means they're having trouble coping with or making sense of their emotional state.

The joke is humorous because it takes the literal meaning of "process" from the AI context and applies it to the emotional context, creating a clever wordplay. The punchline "it was struggling to process its emotions" is a clever double entendre, implying that the AI program is having trouble dealing with its

In [6]:
#2 runnable parallel

from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnableSequence, RunnableParallel


load_dotenv()

model = ChatGroq(model="llama-3.3-70b-versatile")

prompt1 = PromptTemplate(
    template='Generate a tweet about {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a Linkedin post about {topic}',
    input_variables=['topic']
)

parsrer = StrOutputParser()

parallel_chain = RunnableParallel({
    'tweet': prompt1 | model | parsrer,
    'linkedin': prompt2 | model | parsrer
})

result = parallel_chain.invoke({'topic':'AI'})

print(result['tweet'])
print(result['linkedin'])

"AI is revolutionizing the future As machines learn to think, create, and adapt, what possibilities will emerge? Will we see a new era of innovation or unprecedented challenges? Share your thoughts on the future of AI #AI #ArtificialIntelligence #FutureTech"
**The Future is Here: Embracing AI in the Workplace**

As we continue to navigate the ever-evolving landscape of technology, one thing is clear: Artificial Intelligence (AI) is no longer just a buzzword, but a reality that's transforming the way we work and live.

From automating mundane tasks to augmenting human capabilities, AI is revolutionizing industries and creating new opportunities for growth and innovation. But what does this mean for us as professionals?

**Key Takeaways:**

1. **Upskilling and Reskilling**: As AI takes over routine tasks, it's essential to develop skills that complement AI, such as critical thinking, creativity, and problem-solving.
2. **Augmenting Human Capabilities**: AI can enhance our abilities, free

In [8]:
# runnable pass through

from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence, RunnableParallel


load_dotenv()

model = ChatGroq(model="llama-3.3-70b-versatile")

prompt1 = PromptTemplate(
    template='write a joke about {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Explain the following joke - {text}',
    input_variables=['text']
)

parser = StrOutputParser()

joke_chain = RunnableSequence(prompt1 | model | parser)

parralel_chain = RunnableParallel({
    'joke': RunnablePassthrough(),
    'Explaination': RunnableSequence(prompt2 | model | parser)
})

final_chain = RunnableSequence(joke_chain | parralel_chain)

final_chain.invoke({'topic':'singing'})


{'joke': 'Why did the singer bring a ladder to the concert?\n\nBecause she wanted to hit the high notes.',
 'Explaination': 'A classic play on words. This joke is funny because it\'s a clever pun. Here\'s how it works:\n\n* In music, "high notes" refers to the high-pitched sounds that a singer tries to reach when singing a song.\n* However, the phrase "hit the high notes" can also be interpreted literally, as in physically reaching something that\'s high up.\n* The joke sets up the expectation that the singer is trying to achieve something musically (hitting the high notes), but the punchline subverts this expectation by introducing a ladder, which is a physical object used to reach high places.\n* The humor comes from the unexpected twist on the phrase "hit the high notes", implying that the singer needs a ladder to literally reach the high notes, rather than just singing them.\n\nSo, the joke is playing with the double meaning of "high notes" to create a clever and amusing connection

In [12]:
# Runnable Lambda (very important)


from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence, RunnableParallel, RunnableLambda


load_dotenv()

def word_count(text):
    return len(text.split())

model = ChatGroq(model="llama-3.3-70b-versatile")

prompt = PromptTemplate(
    template='write a joke about {topic}',
    input_variables=['topic']
)

parser = StrOutputParser()

joke_chain = RunnableSequence(prompt1 | model | parser)

parralel_chain = RunnableParallel({
    'joke': RunnablePassthrough(),
    'word_count': RunnableLambda(word_count)
})

final_chain = RunnableSequence(joke_chain | parralel_chain)
result = final_chain.invoke({'topic':'singing'})

final_result = """() The joke is: {joke} and the word count of the joke is {word_count}""".format(joke=result['joke'], word_count=result['word_count'])
print(final_result)

() The joke is: Why did the singer bring a ladder to the concert?

Because she wanted to hit the high notes. and the word count of the joke is 18


In [17]:
# runnable branch


from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence, RunnableParallel


load_dotenv()

model = ChatGroq(model="llama-3.3-70b-versatile")

prompt1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Summarize the following text \n {text}',
    input_variables=['text']
)

parser = StrOutputParser()


report_gen_chain = prompt1 | model | parser

branch_chain = RunnableBranch(
    (lambda x: len(x.split())>300, prompt2 | model | parser),
    RunnablePassthrough()
)

final_chain = RunnableSequence(report_gen_chain, branch_chain)

print(final_chain.invoke({'topic':'Russia vs Ukraine'}))

The conflict between Russia and Ukraine is a complex issue with deep historical, cultural, and economic roots. The conflict escalated in 2014 when Ukraine's pro-Russian president, Viktor Yanukovych, was ousted, and Russia annexed Crimea. The conflict has resulted in significant humanitarian, economic, and political consequences, including the deaths of over 13,000 people and the displacement of over 3.5 million.

The conflict has several causes, including:

1. Historical and cultural ties between Russia and Ukraine
2. Economic interests, with Ukraine being a significant economic partner for Russia
3. Geopolitical rivalry between Russia and the West
4. Nationalism and identity-based tensions, including the desire for greater autonomy and self-determination among Ukraine's Russian-speaking population

The conflict has had significant consequences, including:

1. Humanitarian crisis
2. Economic decline in Ukraine
3. Deterioration of relations between Russia and the West
4. Escalation of t

### LCEL :
LCEL stands for "Language Chain Execution Language" and is a domain-specific language designed to define and execute chains of tasks in a structured and efficient manner. LCEL allows users to specify the sequence of tasks, the conditions for branching, and the handling of inputs and outputs in a clear and concise way. By using LCEL, users can create complex workflows that leverage the capabilities of language models and other tools, while also providing features such as error handling, logging, and input/output management to ensure the reliability and maintainability of their applications. LCEL provides a powerful framework for defining and executing language chains, making it easier for users to create sophisticated applications that can handle a variety of tasks and scenarios.